# Notebook 1: Data Exploration

Explore the NTNU Autoferry Sensor Fusion Dataset.

**Sensors:**
- Lidar (ID=1): Active, 2D position (N, E)
- Radar (ID=2): Active, 2D position (N, E)
- IR Camera (ID=3): Passive, bearing only
- EO Camera (ID=4): Passive, bearing only

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from data_loader import SensorDataLoader

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## 1.1 Load Scenario Data

In [ ]:
# Choose a scenario
SCENARIO = 'scenario2'
DATA_PATH = f'../data/sensor_fusion_dataset/{SCENARIO}'

loader = SensorDataLoader(DATA_PATH)
detections = loader.load_all_detections()
ground_truth = loader.load_ground_truth()
ownship = loader.load_ownship()

print(f"Loaded {len(detections)} detection files")
print(f"Ground truth targets: {list(ground_truth.keys())}")
print(f"Ownship trajectory shape: {ownship.shape}")

## 1.2 Sensor Detection Overview

In [ ]:
for sensor_id, df in detections.items():
    print(f"\n=== Sensor {sensor_id} ===")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Time range: {df['time'].min():.1f} - {df['time'].max():.1f}s")
    print(df.head(3))

## 1.3 Plot Scenario Overview

In [ ]:
from visualization.plot_utils import plot_scenario_overview

fig = plot_scenario_overview(detections, ground_truth, ownship, title=f'{SCENARIO} Overview')
plt.show()

## 1.4 Detection Timeline

In [ ]:
from visualization.plot_utils import plot_detection_timeline

fig = plot_detection_timeline(detections, title=f'{SCENARIO} Detection Timeline')
plt.show()

## 1.5 Individual Sensor Plots

In [ ]:
from visualization.plot_utils import plot_all_sensors

fig = plot_all_sensors(detections, ground_truth, title=f'{SCENARIO} All Sensors')
plt.show()

## 1.6 Ground Truth Trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for target_id, gt_df in ground_truth.items():
    ax.plot(gt_df['x_piren'], gt_df['y_piren'], label=f'Target {target_id}', linewidth=2)
    ax.scatter(gt_df['x_piren'].iloc[0], gt_df['y_piren'].iloc[0], marker='o', s=100, zorder=5)

ax.plot(ownship['x_piren'], ownship['y_piren'], 'k--', label='Ownship', linewidth=1)
ax.set_xlabel('East (m)')
ax.set_ylabel('North (m)')
ax.set_title('Ground Truth Trajectories')
ax.legend()
ax.grid(True)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 1.7 Compare Multiple Scenarios

In [ ]:
scenarios = ['scenario2', 'scenario3', 'scenario4']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, scen in zip(axes, scenarios):
    path = f'../data/sensor_fusion_dataset/{scen}'
    ld = SensorDataLoader(path)
    dets = ld.load_all_detections()
    gt = ld.load_ground_truth()
    
    for sid, df in dets.items():
        ax.scatter(df['x_piren'], df['y_piren'], s=1, alpha=0.3, label=f'Sensor {sid}')
    for tid, gdf in gt.items():
        ax.plot(gdf['x_piren'], gdf['y_piren'], 'k-', linewidth=2)
    
    ax.set_title(scen)
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.grid(True)

plt.suptitle('Scenario Comparison', fontsize=14)
plt.tight_layout()
plt.show()